# Modélisation 

### Import des modules 

In [1]:
# Bibliothèques
import pandas as pd
import numpy as np

### Import des données

In [2]:
# Charger les données nettoyées
building_consumption_clean = pd.read_csv('./sources/2016_Building_Energy_Benchmarking_01_cleaned.csv')

#print(f"Dataset original : {building_consumption_clean.shape}")
#print(f"\nAperçu des données :")
#display(building_consumption_clean.head())
#print(f"\nTypes de données :")
#display(building_consumption_clean.dtypes)

---

### Feature Engineering

A réaliser : Enrichir le jeu de données actuel avec de nouvelles features issues de celles existantes. 

---

In [3]:
# 1. FEATURES TEMPORELLES - Âge du bâtiment
building_consumption_clean['Age_batiment'] = 2016 - building_consumption_clean['YearBuilt']

# Catégorisation de l'âge
def categorize_age(age):
    if age < 10:
        return 'Très_récent'
    elif age < 30:
        return 'Récent'
    elif age < 50:
        return 'Ancien'
    else:
        return 'Très_ancien'

building_consumption_clean['Categorie_age'] = building_consumption_clean['Age_batiment'].apply(categorize_age)

print("Features temporelles créées :")
print(f"- Age_batiment: {building_consumption_clean['Age_batiment'].min():.0f} à {building_consumption_clean['Age_batiment'].max():.0f} ans")
print(f"- Catégories d'âge: {building_consumption_clean['Categorie_age'].value_counts().to_dict()}")

Features temporelles créées :
- Age_batiment: 1 à 116 ans
- Catégories d'âge: {'Très_ancien': 748, 'Ancien': 316, 'Récent': 276, 'Très_récent': 101}


In [4]:
# 2. FEATURES DE RATIOS ET INTENSITÉS

# Consommation par étage (avec gestion des 0)
building_consumption_clean['Consommation_par_etage'] = building_consumption_clean['SiteEnergyUse(kWh)'] / (building_consumption_clean['NumberofFloors'] + 1)  # +1 pour éviter division par zéro

# Surface par étage (avec gestion des 0)
building_consumption_clean['Surface_par_etage'] = building_consumption_clean['PropertyGFATotal'] / (building_consumption_clean['NumberofFloors'] + 1)  # +1 pour éviter division par zéro

# Densité énergétique (énergie par m²)
building_consumption_clean['Densite_energetique'] = building_consumption_clean['SiteEnergyUse(kWh)'] / (building_consumption_clean['PropertyGFATotal'] + 1)  # +1 pour éviter division par zéro

# Ratio électricité/gaz
building_consumption_clean['Ratio_elec_gaz'] = building_consumption_clean['Electricity(kWh)'] / (building_consumption_clean['NaturalGas(kWh)'] + 1)  # +1 pour éviter division par zéro

# Part de la surface construite
building_consumption_clean['Part_surface_construite'] = building_consumption_clean['PropertyGFABuilding(s)'] / building_consumption_clean['PropertyGFATotal'].replace(0, 1)

print("\n✓ Features de ratios créées :")
print(f"- Consommation_par_etage: moyenne = {building_consumption_clean['Consommation_par_etage'].mean():.2f}, max = {building_consumption_clean['Consommation_par_etage'].max():.2f}")
print(f"- Surface_par_etage: moyenne = {building_consumption_clean['Surface_par_etage'].mean():.2f}, max = {building_consumption_clean['Surface_par_etage'].max():.2f}")
print(f"- Densite_energetique: moyenne = {building_consumption_clean['Densite_energetique'].mean():.2f}, max = {building_consumption_clean['Densite_energetique'].max():.2f}")
print(f"- Ratio_elec_gaz: moyenne = {building_consumption_clean['Ratio_elec_gaz'].mean():.2f}, max = {building_consumption_clean['Ratio_elec_gaz'].max():.2f}")
print(f"- Part_surface_construite: moyenne = {building_consumption_clean['Part_surface_construite'].mean():.4f}, max = {building_consumption_clean['Part_surface_construite'].max():.4f}")


✓ Features de ratios créées :
- Consommation_par_etage: moyenne = 464088.75, max = 19063477.12
- Surface_par_etage: moyenne = 22966.39, max = 947987.00
- Densite_energetique: moyenne = 19.91, max = 244.53
- Ratio_elec_gaz: moyenne = 458186.57, max = 26395222.00
- Part_surface_construite: moyenne = 0.9320, max = 1.0000


In [5]:
# 3. FEATURES DE TAILLE ET CATÉGORISATION
# Catégoriser la taille du bâtiment
def categorize_size(surface):
    if surface < 50000:
        return 'Petit'
    elif surface < 150000:
        return 'Moyen'
    elif surface < 500000:
        return 'Grand'
    else:
        return 'Très_grand'

building_consumption_clean['Taille_batiment'] = building_consumption_clean['PropertyGFATotal'].apply(categorize_size)

# Catégoriser le nombre d'étages
def categorize_floors(floors):
    if floors <= 3:
        return 'Bas'
    elif floors <= 10:
        return 'Moyen'
    elif floors <= 20:
        return 'Haut'
    else:
        return 'Très_haut'

building_consumption_clean['Hauteur_batiment'] = building_consumption_clean['NumberofFloors'].apply(categorize_floors)

print("\nFeatures de catégorisation créées :")
print(f"- Taille_batiment: {building_consumption_clean['Taille_batiment'].value_counts().to_dict()}")
print(f"- Hauteur_batiment: {building_consumption_clean['Hauteur_batiment'].value_counts().to_dict()}")


Features de catégorisation créées :
- Taille_batiment: {'Petit': 741, 'Moyen': 442, 'Grand': 205, 'Très_grand': 53}
- Hauteur_batiment: {'Bas': 938, 'Moyen': 395, 'Haut': 64, 'Très_haut': 44}


In [6]:
# 4. FEATURES D'INTERACTIONS
# Interaction surface x âge
building_consumption_clean['Surface_x_Age'] = building_consumption_clean['PropertyGFATotal'] * building_consumption_clean['Age_batiment']

# Interaction étages x âge
building_consumption_clean['Etages_x_Age'] = building_consumption_clean['NumberofFloors'] * building_consumption_clean['Age_batiment']

print("\nFeatures d'interactions créées :")
print(f"- Surface_x_Age: valeurs de {building_consumption_clean['Surface_x_Age'].min():.0f} à {building_consumption_clean['Surface_x_Age'].max():.0f}")
print(f"- Etages_x_Age: valeurs de {building_consumption_clean['Etages_x_Age'].min():.0f} à {building_consumption_clean['Etages_x_Age'].max():.0f}")


Features d'interactions créées :
- Surface_x_Age: valeurs de 37873 à 143619736
- Etages_x_Age: valeurs de 0 à 4368


In [7]:
# 5. TRANSFORMATIONS MATHÉMATIQUES (pour distributions asymétriques)
# Log de la consommation énergétique
building_consumption_clean['Log_SiteEnergyUse'] = np.log1p(building_consumption_clean['SiteEnergyUse(kWh)'])  # log1p = log(1+x) pour gérer les valeurs proches de 0

# Racine carrée de la surface
building_consumption_clean['Sqrt_PropertyGFATotal'] = np.sqrt(building_consumption_clean['PropertyGFATotal'])

print("\nTransformations mathématiques créées :")
print(f"- Log_SiteEnergyUse: {building_consumption_clean['Log_SiteEnergyUse'].describe()[['mean', 'std']]}")
print(f"- Sqrt_PropertyGFATotal: {building_consumption_clean['Sqrt_PropertyGFATotal'].describe()[['mean', 'std']]}")


Transformations mathématiques créées :
- Log_SiteEnergyUse: mean    13.686471
std      1.299618
Name: Log_SiteEnergyUse, dtype: float64
- Sqrt_PropertyGFATotal: mean    284.804761
std     178.230104
Name: Sqrt_PropertyGFATotal, dtype: float64


In [8]:
# 6. FEATURES SPÉCIFIQUES AU TYPE DE BÂTIMENT
# Regroupement des types de bâtiments similaires
def group_building_types(prop_type):
    if prop_type in ['Hotel']:
        return 'Hospitality'
    elif prop_type in ['Warehouse', 'Distribution Center', 'Self-Storage Facility']:
        return 'Storage'
    elif prop_type in ['Small- and Mid-Sized Office', 'Large Office', 'Office']:
        return 'Office'
    elif prop_type in ['Retail Store', 'Supermarket / Grocery Store']:
        return 'Retail'
    elif prop_type in ['K-12 School', 'University']:
        return 'Education'
    elif prop_type in ['Hospital', 'Medical Office']:
        return 'Healthcare'
    else:
        return 'Other'

building_consumption_clean['Building_Group'] = building_consumption_clean['PrimaryPropertyType'].apply(group_building_types)

print("\nRegroupement des types de bâtiments :")
print(building_consumption_clean['Building_Group'].value_counts())


Regroupement des types de bâtiments :
Building_Group
Office         449
Other          426
Storage        259
Retail         129
Hospitality     75
Education       55
Healthcare      48
Name: count, dtype: int64


In [9]:
# RÉSUMÉ FINAL
print("\n" + "="*60)
print("RÉSUMÉ DU FEATURE ENGINEERING")
print("="*60)
print(f"\nNombre de features AVANT feature engineering: 10")
print(f"Nombre de features APRÈS feature engineering: {building_consumption_clean.shape[1]}")
print(f"\nNouvelles features créées: {building_consumption_clean.shape[1] - 10}")

print("\n\nListe de toutes les nouvelles features :")
new_features = [col for col in building_consumption_clean.columns if col not in [
    'OSEBuildingID', 'PrimaryPropertyType', 'YearBuilt', 'PropertyGFATotal',
    'PropertyGFABuilding(s)', 'NumberofFloors', 'SiteEnergyUse(kWh)',
    'SiteEUI(kWh/m2)', 'Electricity(kWh)', 'NaturalGas(kWh)',
    'TotalGHGEmissions', 'GHGEmissionsIntensity'
]]

for i, feature in enumerate(new_features, 1):
    print(f"{i}. {feature}")

# Sauvegarder le dataset enrichi
building_consumption_clean.to_csv('./sources/2016_Building_Energy_Benchmarking_02_featured.csv', index=False)
print("\nDataset enrichi sauvegardé dans './sources/2016_Building_Energy_Benchmarking_02_featured.csv'")

# Afficher un aperçu
print("\nAperçu des données enrichies :")
display(building_consumption_clean.head())


RÉSUMÉ DU FEATURE ENGINEERING

Nombre de features AVANT feature engineering: 10
Nombre de features APRÈS feature engineering: 24

Nouvelles features créées: 14


Liste de toutes les nouvelles features :
1. Age_batiment
2. Categorie_age
3. Consommation_par_etage
4. Surface_par_etage
5. Densite_energetique
6. Ratio_elec_gaz
7. Part_surface_construite
8. Taille_batiment
9. Hauteur_batiment
10. Surface_x_Age
11. Etages_x_Age
12. Log_SiteEnergyUse
13. Sqrt_PropertyGFATotal
14. Building_Group

Dataset enrichi sauvegardé dans './sources/2016_Building_Energy_Benchmarking_02_featured.csv'

Aperçu des données enrichies :


,OSEBuildingID,PrimaryPropertyType,YearBuilt,PropertyGFATotal,PropertyGFABuilding(s),NumberofFloors,SiteEnergyUse(kWh),SiteEUI(kWh/m2),Electricity(kWh),NaturalGas(kWh),...,Densite_energetique,Ratio_elec_gaz,Part_surface_construite,Taille_batiment,Hauteur_batiment,Surface_x_Age,Etages_x_Age,Log_SiteEnergyUse,Sqrt_PropertyGFATotal,Building_Group
0,1,Hotel,1927,88434,88434,12,2.117838e+06,261.43999,1.156514e+06,3.740020e+05,...,23.947959,3.092259,1.000000,Moyen,Haut,7870626,1068,14.565907,297.378547,Hospitality
1,2,Hotel,1996,103566,88502,11,2.458260e+06,303.36001,9.504252e+05,1.507514e+06,...,23.735944,0.630458,0.854547,Moyen,Haut,2071320,220,14.714965,321.816718,Hospitality
2,3,Hotel,1969,956110,759392,41,2.127316e+07,307.20000,1.451544e+07,4.376849e+05,...,22.249673,33.164046,0.794252,Très_grand,Très_haut,44937170,1927,16.872957,977.808775,Hospitality
3,5,Hotel,1926,61320,61320,10,1.991296e+06,354.56001,8.115253e+05,5.306872e+05,...,32.473313,1.529194,1.000000,Moyen,Moyen,5518800,900,14.504297,247.628754,Hospitality
4,8,Hotel,1980,175580,113580,18,4.153581e+06,367.36001,1.573449e+06,2.579580e+06,...,23.656209,0.609963,0.646885,Grand,Haut,6320880,648,15.239482,419.022672,Hospitality
